# AF3 to predict structure

## Prepare sequence json file

In [ ]:
import pandas as pd

In [ ]:
com = pd.read_csv('./tf/tf_tf/specific_filtered_1888.csv', index_col=0)
com

In [ ]:
import pandas as pd
import requests
import json
import os
import time
from tqdm import tqdm # Used to display progress bars; install via pip install tqdm first

# --- Configuration Section ---

# 1. Set the NCBI Taxonomy ID of the organism
ORGANISM_ID = "9606" # Human (Homo sapiens)

# 2. Folder name for output JSON files
OUTPUT_DIR = "alphafold3_inputs"


# --- Function Definitions ---

# Global variable to cache queried sequences to avoid repeated network requests
sequence_cache = {}

def get_protein_sequence(gene_name: str, organism_id: str) -> str | None:
    """
    Fetches protein sequence by gene name and organism ID via UniProt API.
    Prioritizes reviewed (Swiss-Prot) sequences.
    """
    if gene_name in sequence_cache:
        return sequence_cache[gene_name]

    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        'query': f'(gene:{gene_name}) AND (organism_id:{organism_id}) AND (reviewed:true)',
        'format': 'fasta',
        'size': 1 
    }

    try:
        time.sleep(0.5) 
        response = requests.get(url, params=params)
        response.raise_for_status()

        fasta_data = response.text
        if not fasta_data:
            print(f"Warning: Protein sequence for gene '{gene_name}' not found.")
            return None

        lines = fasta_data.strip().split('\n')
        sequence = "".join(lines[1:])
        
        if sequence:
            sequence_cache[gene_name] = sequence
            return sequence
        else:
            print(f"Warning: No valid sequence returned for gene '{gene_name}'.")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Error: Network error occurred while requesting sequence for gene '{gene_name}': {e}")
        return None

# ==============================================================================
#  vvv Main Modified Function vvv
# ==============================================================================
def create_alphafold3_json(gene1: str, gene2: str, sequence1: str, sequence2: str, output_path: str):
    """
    Creates and saves the JSON file required by AlphaFold Server API based on gene names and protein sequences.
    """
    # Construct input structure compliant with AlphaFold Server API v1
    af3_input = {
        "name": f"{gene1}_{gene2}",
        "modelSeeds": [1], # Model seeds, default is [1]
        "sequences": [
            {
                "protein": {
                    "id": "A", # Chain A
                    "sequence": sequence1
                }
            },
            {
                "protein": {
                    "id": "B", # Chain B
                    "sequence": sequence2
                }
            }
        ],
        "dialect": "alphafold3",
        "version": 1
    }
    
    try:
        with open(output_path, 'w') as f:
            json.dump(af3_input, f, indent=2)
    except IOError as e:
        print(f"Error: Unable to write to file {output_path}: {e}")
# ==============================================================================
#  ^^^ Main Modified Function ^^^
# ==============================================================================


# --- Main Program ---

def main():
    # --- 1. Load your DataFrame ---
    # Replace this with your own data loading method
    # Example: com = pd.read_csv('your_data.csv')
    
    # For demonstration, we create a sample DataFrame
    # data = {
    #     'tf1': ['TP53', 'MYC', 'JUN', 'INVALID_GENE'],
    #     'tf2': ['MDM2', 'MAX', 'FOS', 'TP53']
    # }
    # com = pd.DataFrame(data)
    # print("Loaded DataFrame example:")
    # print(com)
    # print("-" * 30)

    # --- 2. Create output directory ---
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"Creating output directory: {OUTPUT_DIR}")

    # --- 3. Iterate through DataFrame and generate files ---
    print("Starting sequence retrieval and JSON generation...")
    
    for index, row in tqdm(com.iterrows(), total=com.shape[0], desc="Processing gene pairs"):
        gene1 = row['tf1']
        gene2 = row['tf2']

        if not gene1 or not gene2 or pd.isna(gene1) or pd.isna(gene2):
            print(f"Skipping row {index+1} because gene name is empty.")
            continue

        seq1 = get_protein_sequence(gene1, ORGANISM_ID)
        seq2 = get_protein_sequence(gene2, ORGANISM_ID)

        if seq1 and seq2:
            filename = f"{gene1}_{gene2}.json"
            output_file_path = os.path.join(OUTPUT_DIR, filename)
            # Update function call, passing gene names
            create_alphafold3_json(gene1, gene2, seq1, seq2, output_file_path)
        else:
            print(f"Skipping gene pair ({gene1}, {gene2}) because sequence for one or both could not be retrieved.")
    
    print("-" * 30)
    print("Processing complete! ✅")
    print(f"Generated JSON files have been saved in the '{OUTPUT_DIR}' folder.")


if __name__ == '__main__':
    main()

## AF3 predict

In [ ]:
#!/bin/bash

APPDIR="/path/to/alphafold3_project"  # Replace with your actual path if different
ALPHAFOLD3DIR="$APPDIR/alphafold3"
#HMMER3_BINDIR="/usr/bin" # Path to HMMER binaries (**installed via OS package manager or specify your path**)
HMMER3_BINDIR="/path/to/hmmer/bin/" # Path to Conda binarys (**installed via conda**)
DB_DIR="${APPDIR}/database"
MODEL_DIR="${APPDIR}/model"
WORK_DIR=$(pwd)
OUTPUT_DIR="${WORK_DIR}/output/"
LOG_FILE="${WORK_DIR}/af3_run.log"

CUDA_VISIBLE_DEVICES=7 python ${ALPHAFOLD3DIR}/run_alphafold.py \
    --jackhmmer_binary_path="${HMMER3_BINDIR}/jackhmmer" \
    --nhmmer_binary_path="${HMMER3_BINDIR}/nhmmer" \
    --hmmalign_binary_path="${HMMER3_BINDIR}/hmmalign" \
    --hmmsearch_binary_path="${HMMER3_BINDIR}/hmmsearch" \
    --hmmbuild_binary_path="${HMMER3_BINDIR}/hmmbuild" \
    --db_dir="${DB_DIR}" \
    --model_dir="${MODEL_DIR}" \
    --input_dir="${WORK_DIR}/json_input/" \
    --output_dir="${OUTPUT_DIR}" \
    --buckets="256,512,768,1024,1280,1536,2048,2560,3072,3584,4096,4608,5120" \
    --jax_compilation_cache_dir="$APPDIR/jax_compilation"
    2>&1 | tee -a "${LOG_FILE}"

## Check the contents of all folders

In [ ]:
from pathlib import Path

# --- Please modify the main folder path you want to check here ---
# Windows Example: main_folder_path = r'C:\Users\YourUser\Documents\Projects'
# macOS/Linux Example: main_folder_path = '/path/to/experiments'
main_folder_path = './output/cured/'
# -----------------------------------------

# List to store details of non-compliant folders
# Each element will be a dictionary containing name, count, and list of inner folders
non_compliant_folders = []
folders_checked = 0

path = Path(main_folder_path)

print(f"--- Starting check for folder: {path} ---")

if not path.is_dir():
    print(f"Error: Path '{path}' does not exist or is not a directory.")
else:
    # Iterate through all first-level subfolders under the main folder
    for item in path.iterdir():
        if item.is_dir():
            folders_checked += 1
            
            # Use list comprehension to get names of all inner folders at once
            inner_dirs = [sub_item.name for sub_item in item.iterdir() if sub_item.is_dir()]
            
            # The count of inner folders is the length of this list
            subfolder_count = len(inner_dirs)
            
            # Check if the count is not equal to 5
            if subfolder_count != 5:
                # Record details: name, actual count, and list of inner folder names
                non_compliant_folders.append({
                    "name": item.name,
                    "count": subfolder_count,
                    "subfolders": inner_dirs
                })

    # --- After the loop, report the final detailed results ---
    print(f"\n--- Check Completed ---")
    print(f"Total subfolders checked: {folders_checked}.")

    if not non_compliant_folders:
        if folders_checked > 0:
            print("\nConclusion: ✅ All subfolders contain exactly 5 folders.")
        else:
            print("\nConclusion: ⚠️ No subfolders found in the main directory to check.")
    else:
        print(f"\nConclusion: ❌ Found {len(non_compliant_folders)} non-compliant folders (inner folder count is not 5). Details below:")
        # Iterate through the list and print details for each non-compliant folder
        for folder_info in non_compliant_folders:
            print("-" * 20)
            print(f"  Folder '{folder_info['name']}' contains {folder_info['count']} folders (Expected 5):")
            
            if folder_info['subfolders']:
                # Print names of all its inner folders
                for subfolder_name in folder_info['subfolders']:
                    print(f"    - {subfolder_name}")
            else:
                print("    (This folder does not contain any subfolders)")
        print("-" * 20)

## Computing Interaction Interface

In [ ]:
import pathlib
import os
import re
import pandas as pd
from tqdm import tqdm

# BioPython related libraries
from Bio.PDB import MMCIFParser, NeighborSearch

# --- Configuration Parameters ---
MAIN_FOLDER_PATH = pathlib.Path("./output/spe/")
DISTANCE_THRESHOLD = 5.0
OUTPUT_CSV_FILE = "spe_tf_interaction_residues_summary.csv"

# ---- Core Modification 1: Add amino acid 3-letter to 1-letter mapping dictionary ----
AMINO_ACID_MAP = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLU': 'E', 'GLN': 'Q', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}

def find_interface_from_cif(cif_file_path, dist_thresh):
    """
    Analyzes a single CIF file to identify interacting residues.
    Returns a tuple: (Set of Chain 1 residue strings, Set of Chain 2 residue strings).
    """
    parser = MMCIFParser(QUIET=True)
    try:
        structure = parser.get_structure("protein_complex", cif_file_path)
    except Exception:
        return set(), set()

    model = structure[0]
    chain_ids = [chain.id for chain in model]
    if len(chain_ids) < 2:
        return set(), set()
    
    chain_1_id, chain_2_id = chain_ids[0], chain_ids[1]

    chain1_atoms = [atom for atom in model[chain_1_id].get_atoms()]
    chain2_atoms = [atom for atom in model[chain_2_id].get_atoms()]

    if not chain1_atoms or not chain2_atoms:
        return set(), set()

    # Find residues in Chain 1 close to Chain 2
    ns_chain2 = NeighborSearch(chain2_atoms)
    interface_residues_chain1 = set()
    for atom1 in chain1_atoms:
        if ns_chain2.search(atom1.get_coord(), dist_thresh, 'A'):
            interface_residues_chain1.add(atom1.get_parent())

    # Find residues in Chain 2 close to Chain 1
    ns_chain1 = NeighborSearch(chain1_atoms)
    interface_residues_chain2 = set()
    for atom2 in chain2_atoms:
        if ns_chain1.search(atom2.get_coord(), dist_thresh, 'A'):
            interface_residues_chain2.add(atom2.get_parent())

    # ---- Core Modification 2: Format using the mapping dictionary ----
    # If a non-standard amino acid is encountered, use 'X' as the default value
    set_res1_formatted = {
        f"{AMINO_ACID_MAP.get(res.get_resname(), 'X')}{res.get_id()[1]}" 
        for res in interface_residues_chain1
    }
    set_res2_formatted = {
        f"{AMINO_ACID_MAP.get(res.get_resname(), 'X')}{res.get_id()[1]}" 
        for res in interface_residues_chain2
    }

    return set_res1_formatted, set_res2_formatted

def sort_key_for_seed_folders(folder_path):
    """Helper function to sort sample folders numerically."""
    match = re.search(r'sample-(\d+)$', folder_path.name)
    if match:
        return int(match.group(1))
    return -1

def main():
    if not MAIN_FOLDER_PATH.is_dir():
        print(f"Error: Directory '{MAIN_FOLDER_PATH}' does not exist.")
        return

    tf_pair_dirs = [d for d in MAIN_FOLDER_PATH.iterdir() if d.is_dir()]
    all_results = []

    print(f"Starting analysis of {len(tf_pair_dirs)} TF pairs (Outputting single-letter format)...")
    
    for tf_pair_dir in tqdm(tf_pair_dirs, desc="Processing TF Pairs"):
        # ---- Core Modification 3: Convert folder name (row index) to uppercase ----
        pair_name = tf_pair_dir.name.upper()
        
        current_row = {"TF_Pair": pair_name}
        
        # Find and sort seed directories
        seed_dirs = sorted(
            [d for d in tf_pair_dir.glob("seed-1_sample-*") if d.is_dir()],
            key=sort_key_for_seed_folders
        )
        
        # Iterate through the first 5 samples
        for i in range(5):
            col_name_tf1 = f"TF1_seed{i+1}"
            col_name_tf2 = f"TF2_seed{i+1}"

            tf1_res_str, tf2_res_str = "", ""

            if i < len(seed_dirs):
                seed_dir = seed_dirs[i]
                cif_files = list(seed_dir.glob("*.cif"))
                if cif_files:
                    set_res1, set_res2 = find_interface_from_cif(cif_files[0], DISTANCE_THRESHOLD)
                    
                    if set_res1:
                        # Sort residues numerically (e.g., A2, A10) based on position
                        res_list1_sorted = sorted(list(set_res1), key=lambda r: int(re.findall(r'\d+', r)[0]))
                        tf1_res_str = ", ".join(res_list1_sorted)
                    
                    if set_res2:
                        res_list2_sorted = sorted(list(set_res2), key=lambda r: int(re.findall(r'\d+', r)[0]))
                        tf2_res_str = ", ".join(res_list2_sorted)
                else:
                    tf1_res_str = "CIF file not found"
                    tf2_res_str = "CIF file not found"
            else:
                tf1_res_str = "Sample folder missing"
                tf2_res_str = "Sample folder missing"
            
            current_row[col_name_tf1] = tf1_res_str
            current_row[col_name_tf2] = tf2_res_str
        
        all_results.append(current_row)

    # Convert results to DataFrame
    final_df = pd.DataFrame(all_results)
    final_df = final_df.set_index("TF_Pair")
    final_df.to_csv(OUTPUT_CSV_FILE)
    
    print("\nAnalysis complete!")
    print(f"Results saved to file: '{OUTPUT_CSV_FILE}'")
    print("\nTable Preview:")
    print(final_df.head())

if __name__ == "__main__":
    main()

## Count the number of transcription factors and their amino acid lengths

In [ ]:
import json
import pandas as pd
from pathlib import Path

# --- Configuration Parameters ---
# Path to the input folder
INPUT_FOLDER = Path('./json_input/alphafold3_inputs_1888_spe/')

# --- Main Program ---
def main():
    """
    Main function to execute file traversal, parsing, and DataFrame creation.
    """
    if not INPUT_FOLDER.is_dir():
        print(f"Error: Input folder '{INPUT_FOLDER}' does not exist. Please check the path.")
        return

    # Get all files ending with .json in the folder
    json_files = list(INPUT_FOLDER.glob('*.json'))

    if not json_files:
        print(f"Warning: No JSON files found in folder '{INPUT_FOLDER}'.")
        return

    print(f"Found {len(json_files)} JSON files in '{INPUT_FOLDER}', starting processing...")

    # List to store all protein information
    all_proteins_data = []

    for json_file in json_files:
        try:
            # Open and load the JSON file
            with open(json_file, 'r') as f:
                data = json.load(f)

            # The root of the JSON file has a 'name' field, format like "ATF1_ATF7"
            # We use this as the basis for naming proteins
            base_name = data.get('name', 'Unknown_Pair')
            protein_names = base_name.split('_')
            
            # 'sequences' is a list containing protein objects
            sequences_list = data.get('sequences', [])

            # Iterate through the sequences list, typically containing two proteins
            for i, protein_entry in enumerate(sequences_list):
                # The 'protein' key contains id and sequence
                protein_data = protein_entry.get('protein', {})
                sequence = protein_data.get('sequence', '')
                
                # If there is no explicit protein name in JSON, infer it from the base name
                protein_name = protein_names[i] if i < len(protein_names) else f"{base_name}_p{i+1}"
                
                # Calculate sequence length
                sequence_length = len(sequence)

                # Add extracted information to the list
                all_proteins_data.append({
                    'protein_name': protein_name,
                    'sequence': sequence,
                    'length': sequence_length
                })
        
        except json.JSONDecodeError:
            print(f"Warning: File '{json_file.name}' is not a valid JSON file, skipped.")
        except Exception as e:
            print(f"Error occurred while processing file '{json_file.name}': {e}")

    # --- Create DataFrame ---
    if not all_proteins_data:
        print("Failed to extract protein data from any files.")
        return

    # Convert the list of dictionaries to a DataFrame
    df = pd.DataFrame(all_proteins_data)

    # Remove duplicates potentially caused by filename parsing errors
    df = df.drop_duplicates(subset='protein_name')

    # Set the 'protein_name' column as the index (row names)
    df = df.set_index('protein_name')

    # --- Display Results ---
    print("\n--- Processing Complete ---")
    print("Generated DataFrame Preview:")
    print(df.head())
    
    # Save the DataFrame to a CSV file
    output_filename = 'spe_protein_sequences_and_lengths.csv'
    df.to_csv(output_filename)
    print(f"\nDataFrame successfully saved to '{output_filename}'")


if __name__ == "__main__":
    main()

## Com verification analysis runx1 and RORA, REL, NFKB1

In [ ]:
import os
import re
from Bio.PDB import MMCIFParser, NeighborSearch

In [ ]:
CIF_DIRECTORY = "./com_validation/var/fold_runx1_t147a_rora_seed1/"
DISTANCE_THRESHOLD = 5

folder_name = os.path.basename(os.path.normpath(CIF_DIRECTORY))
OUTPUT_FILE = f"{folder_name}.txt" 

In [ ]:
# --- Main Program ---
def find_interface_from_cif(cif_file_path, dist_thresh):
    """
    Analyzes a single CIF file to identify interacting residues.
    Returns a tuple: (String for printing, Set of Chain 1 residues, Set of Chain 2 residues)
    """
    parser = MMCIFParser(QUIET=True)
    try:
        structure = parser.get_structure("protein_complex", cif_file_path)
    except Exception as e:
        error_msg = f"Unable to parse file {os.path.basename(cif_file_path)}: {e}\n"
        return error_msg, set(), set()

    model = structure[0]
    
    chain_ids = [chain.id for chain in model]
    if len(chain_ids) < 2:
        warn_msg = f"Warning: File {os.path.basename(cif_file_path)} has fewer than 2 chains, cannot analyze interaction.\n"
        return warn_msg, set(), set()
    
    chain_1_id, chain_2_id = chain_ids[0], chain_ids[1]

    chain1_atoms = [atom for atom in model[chain_1_id].get_atoms()]
    chain2_atoms = [atom for atom in model[chain_2_id].get_atoms()]

    if not chain1_atoms or not chain2_atoms:
        warn_msg = f"Warning: Chain {chain_1_id} or {chain_2_id} in file {os.path.basename(cif_file_path)} has no atoms.\n"
        return warn_msg, set(), set()

    # Use NeighborSearch for efficient distance searching
    ns_chain2 = NeighborSearch(chain2_atoms)
    interface_residues_chain1 = set()
    for atom1 in chain1_atoms:
        if ns_chain2.search(atom1.get_coord(), dist_thresh, 'A'):
            interface_residues_chain1.add(atom1.get_parent())

    ns_chain1 = NeighborSearch(chain1_atoms)
    interface_residues_chain2 = set()
    for atom2 in chain2_atoms:
        if ns_chain1.search(atom2.get_coord(), dist_thresh, 'A'):
            interface_residues_chain2.add(atom2.get_parent())

    # Format residue objects into 'RESNAME+RESID' string sets for easy intersection
    set_res1_formatted = {f"{res.get_resname()}{res.get_id()[1]}" for res in interface_residues_chain1}
    set_res2_formatted = {f"{res.get_resname()}{res.get_id()[1]}" for res in interface_residues_chain2}

    # Prepare the output string for printing
    output_str = ""
    output_str += f"--- Analyzing file: {os.path.basename(cif_file_path)} ---\n"
    output_str += f"Interaction analysis based on Chain '{chain_1_id}' and Chain '{chain_2_id}' (threshold < {dist_thresh} Å)\n"
    
    if set_res1_formatted and set_res2_formatted:
        # Sort residues numerically for cleaner output (e.g., extracting numbers from strings)
        res_list1_sorted = sorted(list(set_res1_formatted), key=lambda r: int(re.findall(r'\d+', r)[0]))
        res_list2_sorted = sorted(list(set_res2_formatted), key=lambda r: int(re.findall(r'\d+', r)[0]))
        
        output_str += f"  Interacting residues on Chain {chain_1_id} ({len(res_list1_sorted)} found): {', '.join(res_list1_sorted)}\n"
        output_str += f"  Interacting residues on Chain {chain_2_id} ({len(res_list2_sorted)} found): {', '.join(res_list2_sorted)}\n"
    else:
        output_str += "  No interacting residues found under this threshold.\n"
    
    return output_str, set_res1_formatted, set_res2_formatted


if __name__ == "__main__":
    if not os.path.isdir(CIF_DIRECTORY):
        print(f"Error: Directory '{CIF_DIRECTORY}' does not exist. Please modify the CIF_DIRECTORY variable in the script.")
    else:
        # List to store all analysis report strings
        full_analysis_report = []
        
        # Variables for calculating intersection
        core_intersection_sites = None
        processed_files_for_intersection = []

        # 1. Sort files by name
        cif_files = sorted([f for f in os.listdir(CIF_DIRECTORY) if f.endswith(".cif")])

        print(f"Found {len(cif_files)} CIF files, starting analysis in filename order...")

        # 2. Loop through and process files
        for filename in cif_files:
            cif_path = os.path.join(CIF_DIRECTORY, filename)
            
            # Analyze file and retrieve results
            result_string, sites_chain1, sites_chain2 = find_interface_from_cif(cif_path, DISTANCE_THRESHOLD)
            
            # Store single file analysis report
            full_analysis_report.append(result_string)

            # Update intersection
            if sites_chain1: # Calculate intersection only for files where interaction sites were found
                processed_files_for_intersection.append(filename)
                if core_intersection_sites is None:
                    # If it's the first valid file, assign directly
                    core_intersection_sites = sites_chain1
                else:
                    # Otherwise, update intersection
                    core_intersection_sites.intersection_update(sites_chain1)

        # 3. Generate and append the final intersection report
        summary_header = "\n" + "="*50 + "\n" + "=== Core Interaction Site Analysis Summary ===\n" + "="*50 + "\n"
        
        summary_report = ""
        if core_intersection_sites is not None and core_intersection_sites:
            num_files = len(processed_files_for_intersection)
            summary_report += f"In all {num_files} complexes where interactions were detected, the common core interaction sites on Chain 1 (intersection) are:\n"
            
            # Sort the final intersection results
            sorted_core_sites = sorted(list(core_intersection_sites), key=lambda r: int(re.findall(r'\d+', r)[0]))
            
            summary_report += f"  Core residues ({len(sorted_core_sites)} count): {', '.join(sorted_core_sites)}\n\n"
            summary_report += f"List of files involved in intersection calculation ({num_files} count):\n"
            summary_report += "\n".join([f" - {fname}" for fname in processed_files_for_intersection]) + "\n"
            
        elif not processed_files_for_intersection:
            summary_report += "No valid interaction sites detected in any files, cannot calculate intersection.\n"
        else:
            summary_report += "No common interacting residues on Chain 1 across all complexes with detected interactions (intersection is empty).\n"
            summary_report += f"List of files involved in intersection calculation ({len(processed_files_for_intersection)} count):\n"
            summary_report += "\n".join([f" - {fname}" for fname in processed_files_for_intersection]) + "\n"

        full_analysis_report.append(summary_header)
        full_analysis_report.append(summary_report)

        # 4. Write all results to file and print
        final_report_str = "\n".join(full_analysis_report)
        print(final_report_str)
        
        with open(OUTPUT_FILE, 'w') as f_out:
            f_out.write(final_report_str)
            
        print(f"\nAnalysis complete! All results saved to file: {OUTPUT_FILE}")

## Confirm the identity and intersection of protein sequences used

In [ ]:
tf_seq = pd.read_csv('./output/all_protein_sequences_and_lengths.csv')
tf_seq

In [ ]:
tf_seq.replace('ARNTL', 'BMAL1', inplace=True)

In [ ]:
tf_seq['formatted_sequence'] = tf_seq['sequence'].apply(
    lambda seq: ','.join([f"{i+1}-{aa}" for i, aa in enumerate(seq)])
)

In [ ]:
tf_seq = tf_seq.sort_values('protein_name')
tf_seq

In [ ]:
import pandas as pd

# --- Step 1: Preprocess data for efficient calculation ---

# a. In tf_seq, convert 'formatted_sequence' strings into sets
#    This significantly speeds up subsequent intersection calculations
tf_seq['sequence_set'] = tf_seq['formatted_sequence'].apply(lambda x: set(x.split(',')))

# b. In gene_summary, similarly convert 'MutationList' strings into sets
gene_summary['mutation_set'] = gene_summary['MutationSiteAndAA_MutationList'].apply(lambda x: set(x.split(',')))


# --- Step 2: Map information from gene_summary to tf_seq ---

# a. Create a mapping dictionary from GeneSymbol to mutation set for fast lookup
gene_to_mutations_map = gene_summary.set_index('GeneSymbol')['mutation_set'].to_dict()

# b. Apply this mapping to the 'protein_name' column in tf_seq
#    This finds the corresponding mutation set for each row in tf_seq
#    If the gene does not exist in gene_summary, .get() returns an empty set()
tf_seq['mapped_mutation_set'] = tf_seq['protein_name'].apply(lambda name: gene_to_mutations_map.get(name, set()))


# --- Step 3: Calculate intersection counts and create a new column ---

# a. Define a function to calculate the size of the intersection between two sets
def calculate_intersection_count(row):
    return len(row['sequence_set'].intersection(row['mapped_mutation_set']))

# b. Calculate the intersection count within tf_seq
tf_seq['intersection_count'] = tf_seq.apply(calculate_intersection_count, axis=1)


# --- Step 4: Add intersection counts back to gene_summary ---

# a. Create a mapping dictionary from protein_name to intersection count
gene_to_intersection_count_map = tf_seq.set_index('protein_name')['intersection_count'].to_dict()

# b. Apply this mapping to the 'GeneSymbol' column in gene_summary
gene_summary['intersection_count'] = gene_summary['GeneSymbol'].map(gene_to_intersection_count_map).fillna(0).astype(int)


# --- Cleanup and Output ---

# Remove temporary set columns to keep the final result clean
tf_seq.drop(columns=['sequence_set', 'mapped_mutation_set'], inplace=True)
gene_summary.drop(columns=['mutation_set'], inplace=True)


print("Updated tf_seq DataFrame:")
print(tf_seq)

print("\n" + "="*40 + "\n")

print("Updated gene_summary DataFrame:")
print(gene_summary)

In [ ]:
gene_summary['MutationSiteAndAA_MutationCount']

In [ ]:
tf_seq['MutationSite'] = gene_summary['MutationSiteAndAA_MutationList'].tolist()
tf_seq['MutationCount'] = gene_summary['MutationSiteAndAA_MutationCount'].tolist()

In [ ]:
del tf_seq['sequence']

In [ ]:
tf_seq

In [ ]:
tf_seq.to_csv('./tf_var_sites.csv')

### Determine whether all mutations are in the sequence

In [ ]:
seq = 'MEQKPSKVECGSDPEENSARSPDGKRKRKNGQCSLKTSMSGYIPSYLDKDEQCVVCGDKATGYHYRCITCEGCKGFFRRTIQKNLHPTYSCKYDSCCVIDKITRNQCQLCRFKKCIAVGMAMDLVLDDSKRVAKRKLIEQNRERRRKEEMIRSLQQRPEPTPEEWDLIHIATEAHRSTNAQGSHWKQRRKFLPDDIGQSPIVSMPDGDKVDLEAFSEFTKIITPAITRVVDFAKKLPMFSELPCEDQIILLKGCCMEIMSLRAAVRYDPESDTLTLSGEMAVKREQLKNGGLGVVSDAIFELGKSLSAFNLDDTEVALLQAVLLMSTDRSGLLCVDKIEKSQEAYLLAFEHYVNHRKHNIPHFWPKLLMKVTDLRMIGACHASRFLHMKVECPTELFPPLFLEVFEDQEV'

In [ ]:
mut = tf_seq[tf_seq['protein_name'] == 'THRA']['MutationSite'].item()
mut

In [ ]:
set(mut.split(',')) - set(','.join([f"{i+1}-{aa}" for i, aa in enumerate(seq)]).split(','))

In [ ]:
len(set(mut.split(',')) - set(','.join([f"{i+1}-{aa}" for i, aa in enumerate(seq)]).split(',')))

### Adjust part of the protein sequence and re-predict it

In [ ]:
import pandas as pd

In [ ]:
tf_seq = pd.read_csv('./result/new_seq.csv')
tf_seq

In [ ]:
import os

In [ ]:
all = os.listdir('./json_input/all/')

In [ ]:
all

In [ ]:
def is_gene_pair_match(filename, genes_to_check):
    """
    Checks if either GENE1 or GENE2 from the filename 'GENE1_GENE2.json' 
    exists in the genes_to_check set.
    """
    # a. Safely remove the .json suffix to retrieve the 'GENE1_GENE2' portion
    base_name = os.path.splitext(filename)[0]
    
    # b. Split by underscore
    parts = base_name.split('_')
    
    # c. Check if the format is correct (must consist of exactly two parts)
    if len(parts) != 2:
        return False
        
    gene1, gene2 = parts
    
    # d. Exact match: Check if gene1 or gene2 exists in the set
    if gene1 in genes_to_check or gene2 in genes_to_check:
        return True
    
    return False

# 3. Use list comprehension and this new function to perform filtering
filtered_list = [filename for filename in all if is_gene_pair_match(filename, set(tf_seq['gene']))]

In [ ]:
len(filtered_list)

In [ ]:
import shutil

In [ ]:
import os
import shutil

# --- 3. Iterate through the list and copy files ---

# Counters to track successful and failed file copies
success_count = 0
fail_count = 0

print("\nStarting file copy process...")

for filename in filtered_list:
    # a. Construct the full source file path
    source_path = os.path.join('./json_input/all/', filename)
    
    # b. Construct the full destination file path
    destination_path = os.path.join('./json_input/cured_seq', filename)
    
    try:
        # c. Check if the source file exists
        if os.path.exists(source_path):
            # d. Execute the copy operation
            # shutil.copy2() copies both file content and metadata (e.g., modification time)
            # If you only want to copy the content, you can use shutil.copy()
            shutil.copy2(source_path, destination_path)
            print(f"  [Success] Copied '{filename}' to the destination folder.")
            success_count += 1
        else:
            # Note: Ensure 'source_folder' is defined in your previous code or replace with the hardcoded path
            print(f"  [Warning] Source file '{filename}' does not exist in source folder, skipped.")
            fail_count += 1
            
    except Exception as e:
        # Catch other possible errors, such as permission issues
        print(f"  [Failure] Error occurred while copying '{filename}': {e}")
        fail_count += 1

# --- 4. Print summary report ---

print("\nCopy operation completed.")
print(f"Total: Successfully copied {success_count} files, Failed/Skipped {fail_count} files.")

In [ ]:
import os
import json
import pandas as pd

# b. Define the folder path containing JSON files
json_folder_path = './json_input/cured_seq' # Assuming script is in the same directory as cured_seq folder

# --- 2. Create a mapping dictionary from gene name to sequence ---
#    This will significantly improve lookup efficiency
gene_to_sequence_map = tf_seq.set_index('gene')['seq'].to_dict()

print("Example of the created mapping dictionary:")
print(list(gene_to_sequence_map.items())[:2]) # Print the first two key-value pairs


# --- 3. Ensure the target directory exists ---
if not os.path.isdir(json_folder_path):
    print(f"Error: Folder '{json_folder_path}' does not exist. Please check the path.")
    exit()

# --- 4. Get the list of all JSON files ---
try:
    json_files = [f for f in os.listdir(json_folder_path) if f.endswith('.json')]
    if not json_files:
        print(f"Warning: No .json files found in '{json_folder_path}'.")
        exit()
    print(f"\nFound {len(json_files)} JSON files in the folder. Starting processing...")
except Exception as e:
    print(f"Error: Unable to read folder '{json_folder_path}': {e}")
    exit()

# --- 5. Loop through and modify each JSON file ---

# Counters for success and failure
success_count = 0
fail_count = 0
not_found_genes = set() # Use a set to store missing gene names to avoid duplicate printing

for filename in json_files:
    file_path = os.path.join(json_folder_path, filename)
    
    try:
        # a. Read the JSON file
        with open(file_path, 'r') as f:
            data = json.load(f)

        # Flag to record if the file has been modified
        file_modified = False

        # b. Parse gene names from the 'name' field
        #    e.g., "AHR_MEF2A" -> ['AHR', 'MEF2A']
        genes_in_file = data.get('name', '').split('_')

        if not genes_in_file or not data.get('sequences'):
            print(f"  [Warning] File '{filename}' format is incorrect (missing 'name' or 'sequences' fields), skipped.")
            fail_count += 1
            continue

        # c. Iterate through the sequences list in JSON and perform replacement
        for i, seq_entry in enumerate(data['sequences']):
            if i < len(genes_in_file):
                gene_name_to_replace = genes_in_file[i]
                
                # d. Look up the new sequence from the mapping dictionary
                if gene_name_to_replace in gene_to_sequence_map:
                    new_sequence = gene_to_sequence_map[gene_name_to_replace]
                    
                    # Check if the sequence actually needs updating
                    if seq_entry['protein']['sequence'] != new_sequence:
                        seq_entry['protein']['sequence'] = new_sequence
                        file_modified = True
                else:
                    # If this gene is not found in our DataFrame, record it
                    not_found_genes.add(gene_name_to_replace)

        # e. If the file content was modified, write back to the file
        if file_modified:
            with open(file_path, 'w') as f:
                # indent=2 formats the JSON file for easier reading
                json.dump(data, f, indent=2)
            print(f"  [Success] Updated file '{filename}'.")
            success_count += 1
        else:
            print(f"  [Info] File '{filename}' content is already up-to-date, no update needed.")
            # This case can also be counted as success
            success_count += 1

    except json.JSONDecodeError:
        print(f"  [Failure] File '{filename}' is not a valid JSON file, skipped.")
        fail_count += 1
    except Exception as e:
        print(f"  [Failure] Unknown error occurred while processing file '{filename}': {e}")
        fail_count += 1

# --- 6. Print summary report ---
print("\nProcessing complete.")
print(f"Total: {success_count} files processed successfully, {fail_count} files failed or skipped.")

if not_found_genes:
    print("\nWarning: Sequences for the following genes were not found in your tf_seq DataFrame:")
    for gene in sorted(list(not_found_genes)):
        print(f"  - {gene}")

### Adjusted mutation correspondence rate

In [ ]:
gene_summary = pd.read_csv('./clinvar/tf_var.csv', index_col=0)
gene_summary

In [ ]:
tf_seq = pd.read_csv('./result/old/all_protein_sequences_and_lengths.csv')[['protein_name', 'sequence']]
tf_seq

In [ ]:
tf_seq.replace('ARNTL', 'BMAL1', inplace=True)

In [ ]:
new_seq = pd.read_csv('./result/new_seq.csv')
new_seq.columns = ['protein_name', 'sequence']
new_seq

In [ ]:
tf_seq = pd.concat([tf_seq[~tf_seq['protein_name'].isin(new_seq['protein_name'])], new_seq])
tf_seq

In [ ]:
tf_seq['formatted_sequence'] = tf_seq['sequence'].apply(
    lambda seq: ','.join([f"{i+1}-{aa}" for i, aa in enumerate(seq)])
)

In [ ]:
tf_seq

In [ ]:
tf_seq = tf_seq.sort_values('protein_name')
tf_seq

In [ ]:
import pandas as pd

# --- Step 1: Preprocess data for efficient calculation ---

# a. In tf_seq, convert 'formatted_sequence' strings into sets
#    This significantly speeds up subsequent intersection calculations
tf_seq['sequence_set'] = tf_seq['formatted_sequence'].apply(lambda x: set(x.split(',')))

# b. In gene_summary, similarly convert 'MutationList' strings into sets
gene_summary['mutation_set'] = gene_summary['MutationSiteAndAA_MutationList'].apply(lambda x: set(x.split(',')))


# --- Step 2: Map information from gene_summary to tf_seq ---

# a. Create a mapping dictionary from GeneSymbol to mutation set for fast lookup
gene_to_mutations_map = gene_summary.set_index('GeneSymbol')['mutation_set'].to_dict()

# b. Apply this mapping to the 'protein_name' column in tf_seq
#    This finds the corresponding mutation set for each row in tf_seq
#    If the gene does not exist in gene_summary, .get() returns an empty set()
tf_seq['mapped_mutation_set'] = tf_seq['protein_name'].apply(lambda name: gene_to_mutations_map.get(name, set()))


# --- Step 3: Calculate intersection counts and create a new column ---

# a. Define a function to calculate the size of the intersection between two sets
def calculate_intersection_count(row):
    return len(row['sequence_set'].intersection(row['mapped_mutation_set']))

# b. Calculate the intersection count within tf_seq
tf_seq['intersection_count'] = tf_seq.apply(calculate_intersection_count, axis=1)


# --- Step 4: Add intersection counts back to gene_summary ---

# a. Create a mapping dictionary from protein_name to intersection count
gene_to_intersection_count_map = tf_seq.set_index('protein_name')['intersection_count'].to_dict()

# b. Apply this mapping to the 'GeneSymbol' column in gene_summary
gene_summary['intersection_count'] = gene_summary['GeneSymbol'].map(gene_to_intersection_count_map).fillna(0).astype(int)


# --- Cleanup and Output ---

# Remove temporary set columns to keep the final result clean
tf_seq.drop(columns=['sequence_set', 'mapped_mutation_set'], inplace=True)
gene_summary.drop(columns=['mutation_set'], inplace=True)


print("Updated tf_seq DataFrame:")
print(tf_seq)

print("\n" + "="*40 + "\n")

print("Updated gene_summary DataFrame:")
print(gene_summary)

In [ ]:
gene_summary

In [ ]:
gene_summary['ratio'] = gene_summary['intersection_count']/gene_summary['MutationSiteAndAA_MutationCount']

In [ ]:
gene_summary.to_csv('./result/var_seq_ratio.csv')

## Reorganize the interaction interface

In [ ]:
import pathlib
import os
import re
import pandas as pd
from tqdm import tqdm

# BioPython related libraries
from Bio.PDB import MMCIFParser, NeighborSearch

# --- Configuration Parameters ---
MAIN_FOLDER_PATH = pathlib.Path("./output/cured/")
DISTANCE_THRESHOLD = 5.0
OUTPUT_CSV_FILE = "spe_tf_interaction_residues_summary.csv"

# ---- Core Modification 1: Add amino acid 3-letter to 1-letter mapping dictionary ----
AMINO_ACID_MAP = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLU': 'E', 'GLN': 'Q', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}

def find_interface_from_cif(cif_file_path, dist_thresh):
    """
    Analyzes a single CIF file to identify interacting residues.
    Returns a tuple: (Set of Chain 1 residue strings, Set of Chain 2 residue strings).
    """
    parser = MMCIFParser(QUIET=True)
    try:
        structure = parser.get_structure("protein_complex", cif_file_path)
    except Exception:
        return set(), set()

    model = structure[0]
    chain_ids = [chain.id for chain in model]
    if len(chain_ids) < 2:
        return set(), set()
    
    chain_1_id, chain_2_id = chain_ids[0], chain_ids[1]

    chain1_atoms = [atom for atom in model[chain_1_id].get_atoms()]
    chain2_atoms = [atom for atom in model[chain_2_id].get_atoms()]

    if not chain1_atoms or not chain2_atoms:
        return set(), set()

    # Find residues in Chain 1 that are close to Chain 2
    ns_chain2 = NeighborSearch(chain2_atoms)
    interface_residues_chain1 = set()
    for atom1 in chain1_atoms:
        if ns_chain2.search(atom1.get_coord(), dist_thresh, 'A'):
            interface_residues_chain1.add(atom1.get_parent())

    # Find residues in Chain 2 that are close to Chain 1
    ns_chain1 = NeighborSearch(chain1_atoms)
    interface_residues_chain2 = set()
    for atom2 in chain2_atoms:
        if ns_chain1.search(atom2.get_coord(), dist_thresh, 'A'):
            interface_residues_chain2.add(atom2.get_parent())

    # ---- Core Modification 2: Format using the mapping dictionary ----
    # If a non-standard amino acid is encountered, use 'X' as the default value
    set_res1_formatted = {
        f"{AMINO_ACID_MAP.get(res.get_resname(), 'X')}{res.get_id()[1]}" 
        for res in interface_residues_chain1
    }
    set_res2_formatted = {
        f"{AMINO_ACID_MAP.get(res.get_resname(), 'X')}{res.get_id()[1]}" 
        for res in interface_residues_chain2
    }

    return set_res1_formatted, set_res2_formatted

def sort_key_for_seed_folders(folder_path):
    """Helper function to sort sample folders numerically."""
    match = re.search(r'sample-(\d+)$', folder_path.name)
    if match:
        return int(match.group(1))
    return -1

def main():
    if not MAIN_FOLDER_PATH.is_dir():
        print(f"Error: Directory '{MAIN_FOLDER_PATH}' does not exist.")
        return

    tf_pair_dirs = [d for d in MAIN_FOLDER_PATH.iterdir() if d.is_dir()]
    all_results = []

    print(f"Starting analysis of {len(tf_pair_dirs)} TF pairs (Outputting single-letter format)...")
    
    for tf_pair_dir in tqdm(tf_pair_dirs, desc="Processing TF Pairs"):
        # ---- Core Modification 3: Convert folder name (row index) to uppercase ----
        pair_name = tf_pair_dir.name.upper()
        
        current_row = {"TF_Pair": pair_name}
        
        # Find and sort seed directories
        seed_dirs = sorted(
            [d for d in tf_pair_dir.glob("seed-1_sample-*") if d.is_dir()],
            key=sort_key_for_seed_folders
        )
        
        for i in range(5):
            col_name_tf1 = f"TF1_seed{i+1}"
            col_name_tf2 = f"TF2_seed{i+1}"

            tf1_res_str, tf2_res_str = "", ""

            if i < len(seed_dirs):
                seed_dir = seed_dirs[i]
                cif_files = list(seed_dir.glob("*.cif"))
                if cif_files:
                    set_res1, set_res2 = find_interface_from_cif(cif_files[0], DISTANCE_THRESHOLD)
                    
                    if set_res1:
                        # Sorting logic remains unchanged (sorting by residue number)
                        res_list1_sorted = sorted(list(set_res1), key=lambda r: int(re.findall(r'\d+', r)[0]))
                        tf1_res_str = ", ".join(res_list1_sorted)
                    
                    if set_res2:
                        res_list2_sorted = sorted(list(set_res2), key=lambda r: int(re.findall(r'\d+', r)[0]))
                        tf2_res_str = ", ".join(res_list2_sorted)
                else:
                    tf1_res_str = "CIF file not found"
                    tf2_res_str = "CIF file not found"
            else:
                tf1_res_str = "Sample folder missing"
                tf2_res_str = "Sample folder missing"
            
            current_row[col_name_tf1] = tf1_res_str
            current_row[col_name_tf2] = tf2_res_str
        
        all_results.append(current_row)

    final_df = pd.DataFrame(all_results)
    final_df = final_df.set_index("TF_Pair")
    final_df.to_csv(OUTPUT_CSV_FILE)
    
    print("\nAnalysis complete!")
    print(f"Results saved to file: '{OUTPUT_CSV_FILE}'")
    print("\nTable Preview:")
    print(final_df.head())

if __name__ == "__main__":
    main()

### replace original result

In [ ]:
cured = pd.read_csv('./result/cured_tf_interaction_residues_summary.csv')
cured

In [ ]:
spe = pd.read_csv('./result/old/spe_tf_interaction_residues_summary.csv')
spe

In [ ]:
com = pd.read_csv('./result/old/com_tf_interaction_residues_summary.csv')
com

In [ ]:
spe_cured = cured[cured['TF_Pair'].isin(list(set(cured['TF_Pair']) & set(spe['TF_Pair'])))]
spe_cured

In [ ]:
com_cured = cured[cured['TF_Pair'].isin(list(set(cured['TF_Pair']) & set(com['TF_Pair'])))]
com_cured

In [ ]:
spe_old = spe[~spe['TF_Pair'].isin(list(set(cured['TF_Pair']) & set(spe['TF_Pair'])))]
spe_old

In [ ]:
com_old = com[~com['TF_Pair'].isin(list(set(cured['TF_Pair']) & set(com['TF_Pair'])))]
com_old

In [ ]:
spe = pd.concat([spe_old, spe_cured])
spe

In [ ]:
spe.columns

In [ ]:
import pandas as pd

# Assuming 'spe' is the DataFrame you have already created
# spe = pd.concat([spe_old, spe_cured])

# --- Step 1: Dynamically identify relevant columns for TF1 and TF2 ---
# Filter all columns starting with 'TF1_seed'
tf1_cols = [col for col in spe.columns if col.startswith('TF1_seed')]
# Filter all columns starting with 'TF2_seed'
tf2_cols = [col for col in spe.columns if col.startswith('TF2_seed')]

print("Identified TF1 related columns:", tf1_cols)
print("Identified TF2 related columns:", tf2_cols)
print("\n")


# --- Step 2: Define helper functions to process strings and perform set operations ---
def get_sites_from_string(site_string):
    """Converts a string like 'P43, E50, Q53' into a set {'P43', 'E50', 'Q53'}."""
    # First check if it is a valid string to prevent issues with null values (NaN)
    if not isinstance(site_string, str):
        return set() # Return an empty set if the value is null
    # Split the string and strip whitespace around each element
    return set(site.strip() for site in site_string.split(','))

def calculate_set_operation(row, target_cols, operation='union'):
    """
    Calculates the union or intersection of sets for specified columns in a DataFrame row.
    
    :param row: A row of the DataFrame (a Series object)
    :param target_cols: List of column names to process (e.g., tf1_cols)
    :param operation: 'union' or 'intersection'
    """
    # Extract data from all target columns in this row and convert them to a list of sets using the function above
    list_of_sets = [get_sites_from_string(row[col]) for col in target_cols]
    
    if not list_of_sets:
        return "" # Return empty string if no target columns

    if operation == 'union':
        # set.union(*list_of_sets) calculates the union of multiple sets
        result_set = set.union(*list_of_sets)
    elif operation == 'intersection':
        # set.intersection(*list_of_sets) calculates the intersection of multiple sets
        result_set = set.intersection(*list_of_sets)
    else:
        return ""

    # Convert the final set back to a sorted, comma-separated string for easy viewing
    return ', '.join(sorted(list(result_set)))


# --- Step 3: Apply the function to each row of the DataFrame to calculate new columns ---
print("Calculating unions and intersections, this may take some time...")

# axis=1 means applying the function row-wise
spe['TF1_Union'] = spe.apply(lambda row: calculate_set_operation(row, tf1_cols, 'union'), axis=1)
spe['TF1_Intersection'] = spe.apply(lambda row: calculate_set_operation(row, tf1_cols, 'intersection'), axis=1)

spe['TF2_Union'] = spe.apply(lambda row: calculate_set_operation(row, tf2_cols, 'union'), axis=1)
spe['TF2_Intersection'] = spe.apply(lambda row: calculate_set_operation(row, tf2_cols, 'intersection'), axis=1)

print("Calculation complete!")


# --- Step 4: Display final results ---
print("\nPreview of DataFrame with four new feature columns added:")

# For better display, select only key columns for preview
display_cols = ['TF_Pair', 'TF1_seed1', 'TF1_seed2', 'TF1_Union', 'TF1_Intersection']
print(spe[display_cols].head())

In [ ]:
spe

In [ ]:
spe.to_csv('./result/spe_1888_sites.csv')

In [ ]:
com = pd.concat([com_old, com_cured])
com

In [ ]:
import pandas as pd

# Assuming 'com' is the DataFrame you have already created (renamed from spe in previous context)
# spe = pd.concat([spe_old, spe_cured])

# --- Step 1: Dynamically identify relevant columns for TF1 and TF2 ---
# Filter all columns starting with 'TF1_seed'
tf1_cols = [col for col in com.columns if col.startswith('TF1_seed')]
# Filter all columns starting with 'TF2_seed'
tf2_cols = [col for col in com.columns if col.startswith('TF2_seed')]

print("Identified TF1 related columns:", tf1_cols)
print("Identified TF2 related columns:", tf2_cols)
print("\n")


# --- Step 2: Define helper functions to process strings and perform set operations ---
def get_sites_from_string(site_string):
    """Converts a string like 'P43, E50, Q53' into a set {'P43', 'E50', 'Q53'}."""
    # First check if it is a valid string to prevent issues with null values (NaN)
    if not isinstance(site_string, str):
        return set() # Return an empty set if the value is null
    # Split the string and strip whitespace around each element
    return set(site.strip() for site in site_string.split(','))

def calculate_set_operation(row, target_cols, operation='union'):
    """
    Calculates the union or intersection of sets for specified columns in a DataFrame row.
    
    :param row: A row of the DataFrame (a Series object)
    :param target_cols: List of column names to process (e.g., tf1_cols)
    :param operation: 'union' or 'intersection'
    """
    # Extract data from all target columns in this row and convert them to a list of sets using the function above
    list_of_sets = [get_sites_from_string(row[col]) for col in target_cols]
    
    if not list_of_sets:
        return "" # Return empty string if no target columns

    if operation == 'union':
        # set.union(*list_of_sets) calculates the union of multiple sets
        result_set = set.union(*list_of_sets)
    elif operation == 'intersection':
        # set.intersection(*list_of_sets) calculates the intersection of multiple sets
        result_set = set.intersection(*list_of_sets)
    else:
        return ""

    # Convert the final set back to a sorted, comma-separated string for easy viewing
    return ', '.join(sorted(list(result_set)))


# --- Step 3: Apply the function to each row of the DataFrame to calculate new columns ---
print("Calculating unions and intersections, this may take some time...")

# axis=1 means applying the function row-wise
com['TF1_Union'] = com.apply(lambda row: calculate_set_operation(row, tf1_cols, 'union'), axis=1)
com['TF1_Intersection'] = com.apply(lambda row: calculate_set_operation(row, tf1_cols, 'intersection'), axis=1)

com['TF2_Union'] = com.apply(lambda row: calculate_set_operation(row, tf2_cols, 'union'), axis=1)
com['TF2_Intersection'] = com.apply(lambda row: calculate_set_operation(row, tf2_cols, 'intersection'), axis=1)

print("Calculation complete!")


# --- Step 4: Display final results ---
print("\nPreview of DataFrame with four new feature columns added:")

# For better display, select only key columns for preview
display_cols = ['TF_Pair', 'TF1_seed1', 'TF1_seed2', 'TF1_Union', 'TF1_Intersection']
print(com[display_cols].head())

In [ ]:
com

In [ ]:
com.to_csv('./result/com_339_sites.csv')

# Sorting out TF pathogenic variants

## Overlaps with clinvar

In [ ]:
import pandas as pd

In [ ]:
# '/path/to/clinvar_data/clinvar_info.csv'

In [ ]:
data = pd.read_table(
        './clinvar/variant_summary.txt.gz'
    )

In [ ]:
len(data)

In [ ]:
data.head()

In [ ]:
data.columns

### Extract SNVs that cause protein changes 

In [ ]:
protein_changes = data[data['Name'].str.contains(r'\(p\.', na=False)]
protein_changes_snv = protein_changes[protein_changes['Type'] == 'single nucleotide variant']
protein_changes_snv.head()

In [ ]:
len(protein_changes_snv)

### Extract SNVs of corresponding proteins

In [ ]:
import pandas as pd

In [ ]:
tf = pd.read_csv('./output/old_result/all_protein_sequences_and_lengths.csv')
tf

In [ ]:
set(tf['protein_name']) - set(protein_changes_snv['GeneSymbol'])

In [ ]:
set(['BMAL1']) & set(protein_changes_snv['GeneSymbol'])

In [ ]:
# Replace ARNTL with BMAL1
tf.replace('ARNTL', 'BMAL1', inplace=True)

In [ ]:
set(tf['protein_name']) - set(protein_changes_snv['GeneSymbol'])

In [ ]:
tf_pc_snv = protein_changes_snv[protein_changes_snv['GeneSymbol'].isin(tf['protein_name'])]
tf_pc_snv = tf_pc_snv[tf_pc_snv['Assembly'] == 'GRCh38']
tf_pc_snv

In [ ]:
tf_pc_snv.head().to_csv('./clinvar/var_sample.csv')

In [ ]:
tf_pc_snv.columns

In [ ]:
tf_pc_snv['GeneSymbol'].value_counts()

In [ ]:
tf_pc_snv['ClinicalSignificance'].value_counts()

In [ ]:
tf_pc_snv['PhenotypeList'].value_counts()

In [ ]:
tf_pc_snv[['Name','GeneSymbol','ClinicalSignificance','PhenotypeIDS','PhenotypeList']]

### Summary variant sites

In [ ]:
import pandas as pd
import re

# --- Preparation ---

# 1. Amino acid conversion dictionary (Three-letter to One-letter)
aa_three_to_one = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Glu': 'E', 'Gln': 'Q', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
    'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V',
    'Ter': '*', 'Stop': '*'
}

# --- Step 1: Data Preprocessing ---

def parse_mutation_info(name_string):
    """
    Parses the HGVS nomenclature string and returns a tuple containing 
    (Gene Symbol, "Position-AminoAcid").
    """
    if not isinstance(name_string, str):
        return (None, None)

    # Corrected regular expression to handle hyphens/dashes in gene names
    gene_match = re.search(r'\(([A-Z0-9_.-]+)\)', name_string)
    protein_match = re.search(r'\(p\.([A-Za-z0-9*=_]+)\)', name_string)

    if not gene_match or not protein_match:
        return (None, None)

    gene_symbol = gene_match.group(1)
    p_change = protein_match.group(1)

    # Match the 3-letter amino acid code and the position number
    original_aa_match = re.match(r'([A-Z][a-z]{2})([0-9]+)', p_change)
    if original_aa_match:
        ref_aa_3 = original_aa_match.group(1)
        position = original_aa_match.group(2)
        ref_aa_1 = aa_three_to_one.get(ref_aa_3, '?')
        # Return a tuple for easier splitting later
        return (gene_symbol, f"{position}-{ref_aa_1}")
    
    return (None, None)

# Apply the function and split the result into two columns
# (Assuming 'tf_pc_snv' is your input DataFrame)
parsed_info = tf_pc_snv['Name'].apply(parse_mutation_info).apply(pd.Series)
parsed_info.columns = ['GeneSymbol', 'MutationSiteAndAA']

# Merge parsed results with the original DataFrame (if other columns are needed) or use directly
# Here, we use the parsed new DataFrame directly
temp_df = parsed_info.dropna()


# --- Core Step: Grouping and Multiple Aggregation (groupby.agg) ---

# Define aggregation functions
# 'nunique' is a built-in pandas function used to count the number of unique values
aggregation_functions = {
    'MutationSiteAndAA': [
        ('MutationList', lambda x: ','.join(x.unique())),
        ('MutationCount', 'nunique')
    ]
}

# Perform grouping and aggregation
# as_index=False keeps the grouping key (GeneSymbol) as a column instead of setting it as the index
gene_summary = temp_df.groupby('GeneSymbol', as_index=False).agg(aggregation_functions)

# The DataFrame column names become a MultiIndex; we need to flatten them
gene_summary.columns = ['_'.join(col).strip('_') for col in gene_summary.columns.values]


# --- Output Results ---
print("Summarized mutation list and count by gene:")
print(gene_summary)

In [ ]:
gene_summary

In [ ]:
gene_summary.to_csv('./clinvar/tf_var.csv')

## Organize variant disease information

In [ ]:
var_dis = tf_pc_snv[['Name','GeneSymbol','ClinicalSignificance','PhenotypeIDS','PhenotypeList']]
var_dis

In [ ]:
var_dis.to_csv('./clinvar/tf_var_disease.csv')

In [ ]:
var_dis['PhenotypeIDS'].value_counts()

### Organize mondo

In [ ]:
import pronto
import csv
import time

# --- 0. Define custom mapping rules from ~43 categories to 7 super-categories ---
# This is the core logic of this task
CATEGORY_TO_SUPER_CATEGORY_MAP = {
    # Autoimmune
    'immune system disorder': 'Autoimmune',
    
    # Cancer
    'cancer or benign tumor': 'Cancer',
    
    # Cardiovascular
    'cardiovascular disorder': 'Cardiovascular',
    
    # Metabolic
    'metabolic disease': 'Metabolic',
    'mitochondrial disease': 'Metabolic', # Mitochondrial diseases are fundamentally energy metabolism issues
    
    # Neurological
    'nervous system disorder': 'Neurological',
    'psychiatric disorder': 'Neurological',    # Psychiatric disorders are grouped under Neurological
    'auditory system disorder': 'Neurological', # Auditory system is part of the nervous system
    'disorder of visual system': 'Neurological', # Visual system is part of the nervous system
    'sleep disorder': 'Neurological',          # Sleep disorders are functional issues of the nervous system
    
    # Pulmonary
    'respiratory system disorder': 'Pulmonary',
}


# --- 1. Initialize and Load Ontology ---
print("Starting task: Generating Mondo disease 5-column aggregated classification table...")
print("Loading Mondo ontology file (mondo.obo)...")
start_time = time.time()
try:
    mondo = pronto.Ontology('./clinvar/mondo.obo')
except FileNotFoundError:
    print("Error: 'mondo.obo' file not found in the current directory.")
    exit()
end_time = time.time()
print(f"Mondo ontology loaded! Time elapsed: {end_time - start_time:.2f} seconds.")


# --- 2. Automatically Identify Target Major Categories ---
try:
    root_disease_term = mondo['MONDO:0000001']
    # Find the 'human disease' term among the children of the root
    human_disease_term = next(child for child in root_disease_term.subclasses(distance=1) if child.name == 'human disease')
    # Get the direct subclasses of human disease (the major categories)
    major_categories = list(human_disease_term.subclasses(distance=1, with_self=False))
    print(f"\nSuccessfully identified {len(major_categories)} target major categories.")
except (KeyError, StopIteration):
    print("Error: Unable to find the structure 'MONDO:0000001' -> 'human disease' in the ontology.")
    exit()


# --- 3. Iterate through Major Categories, Find Descendants, and Create 5-Column Mapping ---
print("\nStarting traversal of each major category to aggregate into 7 super-categories...")
all_mappings = []
start_time = time.time()

for i, category_term in enumerate(major_categories):
    print(f"  ({i+1}/{len(major_categories)}) Processing category: '{category_term.name}'...")
    
    # Determine which super-category this major category belongs to; default to 'Other' if not in the map
    super_category_name = CATEGORY_TO_SUPER_CATEGORY_MAP.get(category_term.name, 'Other')
    
    # Get all descendants (including the category itself)
    descendants = category_term.subclasses(with_self=True)
    
    for descendant_term in descendants:
        all_mappings.append([
            descendant_term.name, 
            descendant_term.id, 
            category_term.name, 
            category_term.id,
            super_category_name # Add 5th column: Custom Super Category
        ])

end_time = time.time()
print(f"\nAll mapping relationships generated! Found {len(all_mappings)} records. Time elapsed: {end_time - start_time:.2f} seconds.")


# --- 4. Write All Mappings to CSV File ---
output_filename = 'mondo_aggregated_classification.csv'
print(f"\nWriting results to file: {output_filename}")

try:
    with open(output_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([
            'Disease_Term', 
            'Disease_Term_ID', 
            'Major_Category', 
            'Major_Category_ID',
            'Custom_Super_Category' # New header
        ])
        writer.writerows(all_mappings)
        
    print("\n🎉 Task successfully completed!")
    print(f"Successfully generated final aggregated classification file '{output_filename}'.")

except IOError:
    print(f"Error: Unable to write to file {output_filename}.")

### Organize categories corresponding to mutated diseases

In [ ]:
dis_id = pd.read_csv('./clinvar/mondo_aggregated_classification.csv')
dis_id

In [ ]:
var_dis

In [ ]:
import pandas as pd
import numpy as np # Used for handling potential null values

# --- Assuming you have already loaded these two DataFrames ---
# var_dis = pd.read_csv('path/to/your/variant_data.csv')
# dis_id = pd.read_csv('./mondo_aggregated_classification.csv')


# --- Step 1: Prepare the "Lock" - Create a Unique Disease Classification Lookup Table ---
# Objective: Handle cases in dis_id where one disease ID corresponds to multiple classifications.
# We will merge all classification info for the same ID into one row, separated by semicolons.
print("Step 1: Preparing a unique disease classification lookup table...")

# Define aggregation functions: Deduplicate strings in each group and join them with '; '
agg_funcs = {
    'Disease_Term': 'first', # The name corresponding to the ID is unique, taking the first one is sufficient
    'Major_Category': lambda x: '; '.join(x.unique()),
    'Major_Category_ID': lambda x: '; '.join(x.unique()),
    'Custom_Super_Category': lambda x: '; '.join(x.unique())
}

# Group by Disease_Term_ID and aggregate
dis_lookup = dis_id.groupby('Disease_Term_ID').agg(agg_funcs).reset_index()

print("Lookup table example:")
print(dis_lookup[dis_lookup['Disease_Term'] == 'breast cancer'].to_markdown(index=False))
print("\n")


# --- Step 2 (Corrected Version): Extract and Clean MONDO IDs from var_dis ---
# Objective: Create a new column 'MONDO_ID' in var_dis and fix the double prefix issue.
print("Step 2 (Corrected Version): Extracting and cleaning MONDO IDs from PhenotypeIDS column...")

def extract_and_clean_mondo_id(phenotype_ids_str):
    # Check if input is a valid string
    if not isinstance(phenotype_ids_str, str):
        return np.nan # Return NaN for non-string inputs
    
    # Split the string and look for the ID starting with 'MONDO:'
    for id_part in phenotype_ids_str.split(','):
        if id_part.startswith('MONDO:'):
            # Found the part containing the MONDO ID
            clean_id = id_part.strip()
            
            # --- New Key Correction Step ---
            # If 'MONDO:MONDO:' exists, replace it with a single 'MONDO:'
            if clean_id.startswith('MONDO:MONDO:'):
                clean_id = clean_id.replace('MONDO:MONDO:', 'MONDO:', 1) # Only replace the first occurrence to avoid accidents
            
            return clean_id
    
    return np.nan # Return NaN if not found

# Apply the new, more robust function to the PhenotypeIDS column
var_dis['MONDO_ID'] = var_dis['PhenotypeIDS'].apply(extract_and_clean_mondo_id)

# View the results after cleaning and extraction
print("Example of var_dis table after cleaning and extracting MONDO IDs:")
print(var_dis[['PhenotypeIDS', 'MONDO_ID']].head().to_markdown(index=False))
print("\n")

# --- Step 3: Match and Merge ---
# Objective: Add classification information to the variant data.
print("Step 3: Merging classification info into the variant data table...")

# Use left merge to retain all original variant records
# left_on: Join key for the left table (var_dis)
# right_on: Join key for the right table (dis_lookup)
var_dis_classified = pd.merge(
    var_dis,
    dis_lookup,
    left_on='MONDO_ID',
    right_on='Disease_Term_ID',
    how='left' # 'left' ensures every row in var_dis is retained
)

# Cleanup: Drop temporary 'MONDO_ID' and redundant 'Disease_Term_ID' columns to make the table cleaner
var_dis_classified = var_dis_classified.drop(columns=['MONDO_ID'])

print("🎉 Merge complete! The structure and data examples of the final table are as follows:")

# Show info and first few rows of the final result
print(var_dis_classified.info())
print("\nFinal Result Preview:")
# For better display, we select only a few key columns for preview
display_cols = [
    'Name', 'GeneSymbol', 'ClinicalSignificance', 'PhenotypeList',
    'Disease_Term', 'Major_Category', 'Custom_Super_Category'
]
print(var_dis_classified[display_cols].head().to_markdown(index=False))

In [ ]:
var_dis_classified

In [ ]:
var_dis_classified['Custom_Super_Category'].value_counts()

In [ ]:
var_dis_classified.to_csv('./clinvar/tf_var_disease_mondo.csv')

# Correspond to the mutation of the interaction interface

## Interaction

In [ ]:
import pandas as pd

In [ ]:
com_site = pd.read_csv('./result/com_339_sites.csv', index_col=0)[['TF_Pair', 'TF1_Union',	'TF1_Intersection',	'TF2_Union', 'TF2_Intersection']]
com_site

In [ ]:
import numpy as np

In [ ]:
import numpy as np
import pandas as pd

# --- Step 1: One-step conversion from wide table to tidy long format (TF, Site) ---

# .reset_index() ensures 'TF_Pair' is a column, not an index
# .melt() converts the wide format to long format
# .dropna() immediately drops rows with no site information
# .assign() is a powerful tool for method chaining, creating multiple new columns simultaneously
tidy_df = com_site.reset_index().melt(
    id_vars=['TF_Pair'],
    value_vars=['TF1_Intersection', 'TF2_Intersection'],
    var_name='Source_Column',
    value_name='Site_String'
).dropna(subset=['Site_String']).assign(
    # Determine if it is TF1 or TF2 based on the source column name, and extract the corresponding TF name from TF_Pair
    TF = lambda dfx: np.where(
        dfx['Source_Column'].str.contains('TF1'), 
        dfx['TF_Pair'].str.split('_').str[0], 
        dfx['TF_Pair'].str.split('_').str[1]
    ),
    # Split the site string into a list by comma and optional whitespace
    Site = lambda dfx: dfx['Site_String'].str.split(',\s*')
).explode('Site')[['TF', 'Site']].drop_duplicates()


# --- Step 2: Complete final aggregation in a single line ---

# .groupby('TF') groups by Transcription Factor
# .agg() applies a lambda function: deduplicate (unique), sort (sorted), and then join the sites for each group
final_summary_df = tidy_df.groupby('TF')['Site'].agg(
    lambda sites: ', '.join(sorted(sites.unique()))
).reset_index(name='All_Sites')


# --- Display final results ---
print("🎉 Task completed! Final aggregated results below:")
print(final_summary_df.head())

In [ ]:
final_summary_df.to_csv('./test.csv')

## Variants

In [ ]:
tf_var = pd.read_csv('./clinvar/tf_var_disease_mondo.csv',index_col=0)
tf_var

In [ ]:
tf_var['Custom_Super_Category'].value_counts()

In [ ]:
import pandas as pd
import re  # Import regular expression module
import numpy as np  # Used for handling potential null/missing values

# --- Assuming 'tf_var' is the DataFrame you have already loaded ---
# tf_var = pd.read_csv('./clinvar/tf_var_disease_mondo.csv', index_col=0)


# --- Step 1: Define a dictionary mapping three-letter amino acid codes to one-letter codes ---
AA_MAP = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Glu': 'E', 'Gln': 'Q', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
    'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V',
    'Ter': '*'  # 'Ter' stands for Termination (stop codon), usually represented by an asterisk
}


# --- Step 2: Write a function to extract and convert protein site information ---
def extract_protein_site(name_string):
    """
    Extracts simplified protein site information (e.g., 'C261') from HGVS nomenclature strings.
    """
    # First, check if input is a valid string to prevent issues with null values (NaN)
    if not isinstance(name_string, str):
        return np.nan

    # Regex explanation:
    # p\.         - Matches the literal "p."
    # ([A-Z][a-z]{2}) - 1st Capture Group: Matches one uppercase followed by two lowercase letters (three-letter code)
    # (\d+)       - 2nd Capture Group: Matches one or more digits (position number)
    pattern = re.compile(r'p\.([A-Z][a-z]{2})(\d+)')
    
    match = pattern.search(name_string)
    
    if match:
        three_letter_code = match.group(1) # Get content of group 1 (e.g., 'Cys')
        position = match.group(2)          # Get content of group 2 (e.g., '261')
        
        # Look up the corresponding one-letter code in the map
        one_letter_code = AA_MAP.get(three_letter_code)
        
        if one_letter_code:
            # If code found, format the final string
            return f"{one_letter_code}{position}"
            
    # Return NaN if no match found or three-letter code not in dictionary
    return np.nan


# --- Step 3: Apply function to 'Name' column to create new 'site' column ---
print("Extracting protein site info from 'Name' column and creating 'site' column...")

tf_var['site'] = tf_var['Name'].apply(extract_protein_site)

print("Extraction complete!")


# --- Step 4: Display results for verification ---
print("\nPreview of DataFrame with new 'site' column:")

# Select only key columns for better preview visualization
display_cols = ['Name', 'GeneSymbol', 'site']
print(tf_var[display_cols].head())

# Check for rows where extraction failed (site column is null)
print(f"\nNumber of rows where extraction failed (result is null): {tf_var['site'].isnull().sum()}")

In [ ]:
tf_var

In [ ]:
pd.DataFrame(tf_var['GeneSymbol'].value_counts()).to_csv('./test.csv')

## Integrate

In [ ]:
import pandas as pd
import numpy as np

# --- Assuming final_summary_df and tf_var are DataFrames you have already loaded ---
# final_summary_df = ...
# tf_var = ...
# And tf_var already has a column named 'site'

# --- Step 1: Create an efficient lookup dictionary (Same as before) ---
print("Step 1: Creating an efficient lookup dictionary from tf_var...")

tf_var['Major_Category'] = tf_var['Major_Category'].astype(str).fillna('')
tf_var['Custom_Super_Category'] = tf_var['Custom_Super_Category'].astype(str).fillna('')
lookup_dict = {}

for _, row in tf_var.dropna(subset=['GeneSymbol', 'site']).iterrows():
    key = (row['GeneSymbol'], row['site'])
    major_cats = {cat.strip() for cat in row['Major_Category'].split(';') if cat.strip()}
    super_cats = {cat.strip() for cat in row['Custom_Super_Category'].split(';') if cat.strip()}
    
    if key in lookup_dict:
        lookup_dict[key]['major'].update(major_cats)
        lookup_dict[key]['super'].update(super_cats)
    else:
        lookup_dict[key] = {'major': major_cats, 'super': super_cats}

print(f"Lookup dictionary created, containing {len(lookup_dict)} unique (Gene, Site) entries.\n")


# --- Step 2 (Enhanced): Define function to collect classifications and overlapping sites simultaneously ---
print("Step 2 (Enhanced): Iterating through summary table to collect classifications and overlapping sites...")

def find_matching_info(row, lookup):
    """
    For each row in the summary table, find corresponding classifications for all its sites
    and identify the specific sites that overlap.
    """
    tf_name = row['TF']
    sites_string = row['All_Sites']
    
    if not isinstance(sites_string, str):
        # Return three empty strings, corresponding to the three new columns
        return pd.Series(['', '', '']) 
        
    sites_list = {site.strip() for site in sites_string.split(',')}
    
    # Initialize three sets
    found_major_categories = set()
    found_super_categories = set()
    overlapping_sites = set() # <--- New set to store overlapping sites
    
    # Iterate through all sites for this TF
    for site in sites_list:
        match = lookup.get((tf_name, site))
        if match:
            # If a match is found, update classification sets
            found_major_categories.update(match['major'])
            found_super_categories.update(match['super'])
            # And add this specific site to the overlapping sites set
            overlapping_sites.add(site) # <--- New operation
            
    # Convert final sets to sorted, comma-separated strings
    major_str = ', '.join(sorted(list(found_major_categories)))
    super_str = ', '.join(sorted(list(found_super_categories)))
    overlap_str = ', '.join(sorted(list(overlapping_sites))) # <--- New string conversion
    
    # Return a Series containing three elements
    return pd.Series([major_str, super_str, overlap_str])


# --- Step 3: Apply the function and create three new columns ---
# Apply function row-wise and unpack the returned Series into three new columns
new_columns = ['Major_Category', 'Custom_Super_Category', 'Overlapping_Sites']
final_summary_df[new_columns] = final_summary_df.apply(
    lambda row: find_matching_info(row, lookup_dict),
    axis=1
)

print("🎉 Annotation complete!")


# --- Step 4: Display final results ---
print("\nPreview of final_summary_df with new classification and overlapping site columns:")

display_cols = ['TF', 'All_Sites', 'Major_Category', 'Custom_Super_Category', 'Overlapping_Sites']
# Using ZFP57 as an example again
print(final_summary_df[final_summary_df['TF'] == 'ZFP57'][display_cols].to_markdown(index=False))

# View the head of the full table
print("\nFull table head preview:")
print(final_summary_df[display_cols].head())

In [ ]:
final_summary_df

In [ ]:
final_summary_df.to_csv('./common.csv')

## Export specific hematopoietic variants

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

# --- Assuming final_summary_df and tf_var are DataFrames you have already loaded ---
# final_summary_df = ...
# tf_var = ...
# And tf_var already has a column named 'site'

# --- Step 1: Create an efficient lookup dictionary (Same as before) ---
print("Step 1: Creating an efficient lookup dictionary from tf_var...")

# Ensure category columns are strings and handle NaNs
tf_var['Major_Category'] = tf_var['Major_Category'].astype(str).fillna('')
tf_var['Custom_Super_Category'] = tf_var['Custom_Super_Category'].astype(str).fillna('')
lookup_dict = {}

# Iterate through tf_var to populate the dictionary
for _, row in tf_var.dropna(subset=['GeneSymbol', 'site']).iterrows():
    key = (row['GeneSymbol'], row['site'])
    # Split categories by semicolon and strip whitespace
    major_cats = {cat.strip() for cat in row['Major_Category'].split(';') if cat.strip()}
    super_cats = {cat.strip() for cat in row['Custom_Super_Category'].split(';') if cat.strip()}
    
    if key in lookup_dict:
        lookup_dict[key]['major'].update(major_cats)
        lookup_dict[key]['super'].update(super_cats)
    else:
        lookup_dict[key] = {'major': major_cats, 'super': super_cats}

print(f"Lookup dictionary created, containing {len(lookup_dict)} unique (Gene, Site) entries.\n")


# --- Step 2 (Enhanced): Define function to collect classifications and overlapping sites (Same as before) ---
print("Step 2 (Enhanced): Iterating through summary table to collect classifications and overlapping sites...")

def find_matching_info(row, lookup):
    tf_name = row['TF']
    sites_string = row['All_Sites']
    
    if not isinstance(sites_string, str):
        return pd.Series(['', '', '']) 
        
    sites_list = {site.strip() for site in sites_string.split(',')}
    
    found_major_categories = set()
    found_super_categories = set()
    overlapping_sites = set()
    
    for site in sites_list:
        match = lookup.get((tf_name, site))
        if match:
            found_major_categories.update(match['major'])
            found_super_categories.update(match['super'])
            overlapping_sites.add(site)
            
    major_str = ', '.join(sorted(list(found_major_categories)))
    super_str = ', '.join(sorted(list(found_super_categories)))
    overlap_str = ', '.join(sorted(list(overlapping_sites)))
    
    return pd.Series([major_str, super_str, overlap_str])


# --- Step 3: Apply the function and create three new columns (Same as before) ---
new_columns = ['Major_Category', 'Custom_Super_Category', 'Overlapping_Sites']
final_summary_df[new_columns] = final_summary_df.apply(
    lambda row: find_matching_info(row, lookup_dict),
    axis=1
)

print("🎉 Annotation complete!")


# ======================= [New Output Module - Precise Version] =======================
# --- New Step: Filter and output molecules and sites related to 'hematologic disorder' found in [Overlapping Sites] ---
print("\n--- Identifying molecules and sites related to 'hematologic disorder' from [Overlapping Sites] ---")

hematologic_hits = defaultdict(set)

# Iterate through the annotated final_summary_df
for _, row in final_summary_df.iterrows():
    tf = row['TF']
    overlapping_sites_str = row['Overlapping_Sites']
    
    # Ensure the Overlapping_Sites column is not empty
    if not overlapping_sites_str or pd.isna(overlapping_sites_str):
        continue
        
    # Split the overlapping sites string into a list
    overlapping_sites_list = {site.strip() for site in overlapping_sites_str.split(',')}
    
    # Iterate through [each overlapping site] of this molecule
    for site in overlapping_sites_list:
        # Use lookup_dict to check the classification of this specific (Molecule, Overlapping Site) pair
        match = lookup_dict.get((tf, site))
        if match and 'hematologic disorder' in match['major']:
            # If the classification includes 'hematologic disorder', record it
            hematologic_hits[tf].add(site)

print("Identification complete. The following are [Overlapping] molecules and sites related to 'hematologic disorder':")
if not hematologic_hits:
    print("No molecule/site combinations related to 'hematologic disorder' were found in [Overlapping Sites].")
else:
    # Sort by molecule name alphabetically and print results
    for tf, sites in sorted(hematologic_hits.items()):
        sites_str = ', '.join(sorted(list(sites)))
        print(f"Molecule (TF): {tf:<10} | Related Overlapping Sites: {sites_str}")
# =================================================================


# --- Step 4: Display final results (Same as before) ---
print("\nPreview of final_summary_df with new classification and overlapping site columns:")

display_cols = ['TF', 'All_Sites', 'Major_Category', 'Custom_Super_Category', 'Overlapping_Sites']
# Using ZFP57 as an example again
print(final_summary_df[final_summary_df['TF'] == 'ZFP57'][display_cols].to_markdown(index=False))

# View the head of the full table
print("\nFull table head preview:")
print(final_summary_df[display_cols].head())

## spe

In [ ]:
spe_site = pd.read_csv('./result/spe_1888_sites.csv', index_col=0)[['TF_Pair','TF1_Intersection','TF2_Intersection']].sort_values('TF_Pair')
spe_site

In [ ]:
cell = pd.read_csv('../get/tf/tf_tf/specific_filtered_1888.csv', index_col=0).sort_values('tf_tf')
cell

In [ ]:
spe_site['cell_type'] = cell['cell_type'].tolist()
spe_site

In [ ]:
list_of_dataframes = [group for _, group in spe_site.groupby('cell_type')]

if list_of_dataframes:
    print("(cell_type: {})：".format(list_of_dataframes[0]['cell_type'].iloc[0]))
    print(list_of_dataframes[0].head())

In [ ]:
list_of_dataframes

In [ ]:
len(list_of_dataframes)

In [ ]:
new_columns = ['Major_Category', 'Custom_Super_Category', 'Overlapping_Sites']

for i in range(14):
    print(i)
    tidy_df = list_of_dataframes[i].reset_index().melt(
    id_vars=['TF_Pair'],
    value_vars=['TF1_Intersection', 'TF2_Intersection'],
    var_name='Source_Column',
    value_name='Site_String'
    ).dropna(subset=['Site_String']).assign(
    TF = lambda dfx: np.where(
        dfx['Source_Column'].str.contains('TF1'), 
        dfx['TF_Pair'].str.split('_').str[0], 
        dfx['TF_Pair'].str.split('_').str[1]
    ),
    Site = lambda dfx: dfx['Site_String'].str.split(',\s*')
    ).explode('Site')[['TF', 'Site']].drop_duplicates()

    final_summary_df = tidy_df.groupby('TF')['Site'].agg(
        lambda sites: ', '.join(sorted(sites.unique()))
    ).reset_index(name='All_Sites')
    final_summary_df[new_columns] = final_summary_df.apply(
            lambda row: find_matching_info(row, lookup_dict),
            axis=1
    )
    final_summary_df.to_csv('./result/spe_final/'+list_of_dataframes[i]['cell_type'].tolist()[0]+'.csv')

In [ ]:
spe_site['cell_type'].value_counts()

In [ ]:
spe_site[spe_site['cell_type'] == 'monocytes']

# Visual

In [ ]:
import pandas as pd

## Original draw

In [ ]:
filtered_df = pd.read_csv('./tf/tf_tf/common_filtered_339.csv', index_col=0)
filtered_df

In [ ]:
# Get the unique TFs from 'tf1' and 'tf2' columns
unique_tfs = pd.unique(filtered_df[['tf1', 'tf2']].values.ravel('K'))

# Count the number of unique TFs
num_unique_tfs = len(unique_tfs)
num_unique_tfs

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Set font and PDF output parameters
plt.rcParams['pdf.fonttype'] = 42      # Ensure text is embedded as TrueType fonts
plt.rcParams['ps.fonttype'] = 42       # If PostScript output is required

# Construct the graph
G = nx.from_pandas_edgelist(filtered_df, "tf1", "tf2", edge_attr="score")

# Calculate node centrality
centrality = nx.degree_centrality(G)

# Normalize node sizes (reduce range to avoid overlap)
node_sizes = np.interp(
    list(centrality.values()),
    (min(centrality.values()), max(centrality.values())),
    (100, 800)
)

# Edge color mapping
scores = [d['score'] for (_, _, d) in G.edges(data=True)]
norm = plt.Normalize(vmin=min(scores), vmax=max(scores))
edge_colors = cm.plasma(norm(scores))

# Improve layout: increase k value or use kamada_kawai
pos = nx.spring_layout(G, k=2.0, seed=42)
# pos = nx.kamada_kawai_layout(G)  # Alternative layout option

fig, ax = plt.subplots(figsize=(12, 10))

# Draw nodes
nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color="skyblue",
    alpha=0.9,
    edgecolors="black",
    ax=ax
)

# Draw edges
nx.draw_networkx_edges(
    G, pos,
    edge_color=edge_colors,
    width=np.interp(scores, (min(scores), max(scores)), (0.5, 2)),
    alpha=0.3,
    ax=ax
)

# Labels (Show top 50, add offset to avoid overlapping nodes)
top_nodes = sorted(centrality, key=centrality.get, reverse=True)[:50]
for n in top_nodes:
    x, y = pos[n]
    ax.text(
        x, y+0.05,  # Shift upwards
        n,
        fontsize=9,
        ha="center", va="center",
        bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1)  # White background to prevent overlap with edges
    )

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cm.plasma, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("Edge Score")

ax.set_title("Improved PPI Network (Reduced Overlap)", fontsize=14)
ax.axis("off")
# plt.savefig("ppi_network.pdf", format="pdf", bbox_inches="tight")

plt.show()

In [ ]:
dis = pd.read_csv('../af3/result/com_final.csv', index_col=0)
dis['Overlapping_Sites_Count'] = dis['Overlapping_Sites'].str.split(', ').str.len().fillna(0).astype(int)
dis

In [ ]:
set(unique_tfs) - set(dis['TF'])

## Add disease information

In [ ]:
import pandas as pd
from collections import Counter
import numpy as np
def parse_categories(category_string):
    if pd.isna(category_string) or category_string.lower() == 'nan':
        return Counter()
    categories = [cat.strip() for cat in category_string.split(',')]
    valid_categories = [cat for cat in categories if cat.lower() != 'nan' and cat]
    return Counter(valid_categories)

dis['Category_Counts'] = dis['Major_Category'].apply(parse_categories)

In [ ]:
dis

In [ ]:
dis.to_csv('./tf/tf_tf/com_tf_dis.csv')

In [ ]:
dis['Overlapping_Sites_Count'].min()

In [ ]:
dis['Overlapping_Sites_Count'].max()

In [ ]:
all_categories = set()
for counter in dis['Category_Counts']:
    all_categories.update(counter.keys())

colors = cm.get_cmap('tab20', len(all_categories)) 
category_color_map = {category: colors(i) for i, category in enumerate(all_categories)}

default_node_color = 'lightgrey'

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.patches as mpatches  # Used for the legend

# Assuming 'dis' and 'filtered_df' are already defined
# ... (Your data loading and preprocessing code) ...

# ==============================================================================
# Steps 1 & 2: Data Preprocessing and Color Mapping (Kept unchanged)
def parse_categories(category_string):
    if pd.isna(category_string) or category_string.lower() == 'nan':
        return Counter()
    # Split by comma and strip whitespace
    categories = [cat.strip() for cat in category_string.split(',')]
    # Filter out empty strings or 'nan' strings
    valid_categories = [cat for cat in categories if cat.lower() != 'nan' and cat]
    return Counter(valid_categories)

# Apply parsing to the dataframe
dis['Category_Counts'] = dis['Major_Category'].apply(parse_categories)

# Collect all unique categories
all_categories = set()
for counter in dis['Category_Counts']:
    all_categories.update(counter.keys())

# Generate color map
if all_categories:
    colors = cm.get_cmap('tab20', len(all_categories))
    category_color_map = {category: colors(i) for i, category in enumerate(all_categories)}
else:
    category_color_map = {}
default_node_color = 'lightgrey'
# ==============================================================================

# Set fonts to ensure compatibility with PDF output (embedding TrueType fonts)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Construct the graph
G = nx.from_pandas_edgelist(filtered_df, "tf1", "tf2", edge_attr="score")

# --- Node Size/Radius Settings (Kept unchanged) ---
# Map TFs to their overlapping sites count
tf_to_count_map = dis.set_index('TF')['Overlapping_Sites_Count']
# Align with graph nodes, fill missing with 0
raw_node_sizes = tf_to_count_map.reindex(G.nodes(), fill_value=0)

# Calculate radius interpolation
min_val, max_val = raw_node_sizes.min(), raw_node_sizes.max()
if min_val == max_val:
    node_radii = pd.Series([0.05 if size > 0 else 0 for size in raw_node_sizes], index=raw_node_sizes.index)
else:
    radii_values = np.interp(raw_node_sizes, (min_val, max_val), (0.02, 0.1))
    node_radii = pd.Series(radii_values, index=raw_node_sizes.index)

# --- Plotting Settings ---
# Calculate layout
pos = nx.spring_layout(G, k=2.0, seed=42)
fig, ax = plt.subplots(figsize=(16, 12))

# --- Draw Edges ---
scores = [d['score'] for (_, _, d) in G.edges(data=True)]
nx.draw_networkx_edges(
    G, pos,
    edge_color='#1976d2',
    width=np.interp(scores, (min(scores), max(scores)), (0.5, 2)),
    alpha=0.3,
    ax=ax
)

# --- Draw Nodes as Pie Charts (Kept unchanged) ---
# Create a mapping for easy access inside the loop
tf_to_categories = dis.set_index('TF')['Category_Counts']

for node in G.nodes():
    x, y = pos[node]
    radius = node_radii.get(node, 0.01)
    categories = tf_to_categories.get(node)

    # Determine pie chart data
    if not categories:
        pie_labels = ['default']
        pie_sizes = [1]
        pie_colors = [default_node_color]
    else:
        pie_labels = list(categories.keys())
        pie_sizes = list(categories.values())
        pie_colors = [category_color_map[label] for label in pie_labels]

    # Draw pie chart using inset_axes
    pie_ax = ax.inset_axes([x - radius, y - radius, 2 * radius, 2 * radius],
                           transform=ax.transData)
    pie_ax.pie(pie_sizes, colors=pie_colors, wedgeprops={'edgecolor': 'black', 'linewidth': 0.5})

# --- Draw Labels (Kept unchanged) ---
# Select top 50 largest nodes and any node that has a category assigned
top_50_set = set(raw_node_sizes.nlargest(50).index)
nodes_with_category_set = set(dis[pd.notna(dis['Custom_Super_Category'])]['TF'])
final_nodes_to_label = top_50_set.union(nodes_with_category_set)

for n in final_nodes_to_label:
    if n in pos:
        x, y = pos[n]
        ax.text(
            x, y + node_radii.get(n, 0.02) * 1.1, # Shift text slightly above the node
            n,
            fontsize=10,
            ha="center", va="bottom",
            color='black',
            zorder=10,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1)
        )

# ======================= [Key Modification: Sorted Legend] =======================
# --- Create and Add Legend ---
if category_color_map:
    # 1. Get (key, value) pairs from the dictionary and sort by key (category label)
    sorted_legend_items = sorted(category_color_map.items(), key=lambda item: item[0])
    
    # 2. Create list of legend_patches based on sorted results
    legend_patches = [mpatches.Patch(color=color, label=label) for label, color in sorted_legend_items]
    
    # 3. Create the legend
    ax.legend(handles=legend_patches, title="Categories", bbox_to_anchor=(1.05, 1), loc='upper left')
# =====================================================================

ax.set_title("PPI Network (Nodes as Pie Charts by Category)", fontsize=16)
ax.axis("off")
fig.tight_layout()

plt.savefig("ppi_network.pdf", format="pdf", bbox_inches="tight")
plt.show()

## Check if the node needs to be removed

In [ ]:
import pandas as pd

In [ ]:
dis = pd.read_csv('../af3/result/com_final.csv', index_col=0)
dis['Overlapping_Sites_Count'] = dis['Overlapping_Sites'].str.split(', ').str.len().fillna(0).astype(int)
dis = dis[['TF', 'Overlapping_Sites_Count']]
dis

In [ ]:
new_tfs = {
    'TF': ['ERG', 'FLI1', 'NFIX', 'RUNX3', 'STAT3', 'TGIF2'],
    'Overlapping_Sites_Count': [0, 0, 0, 0, 0, 0]
}
new_rows = pd.DataFrame(new_tfs)

In [ ]:
dis_updated = pd.concat([dis, new_rows], ignore_index=True)

In [ ]:
dis_updated

In [ ]:
filtered_df = pd.read_csv('./tf/tf_tf/common_filtered_339.csv', index_col=0)
filtered_df

In [ ]:
import pandas as pd
import numpy as np # Import numpy for data processing

# --- Step 1: Create an efficient TF -> Count mapping ---
# Set the 'TF' column as the index, then select the 'Overlapping_Sites_Count' column to get a Series.
# This acts like a dictionary; you can retrieve the count value directly using the TF name, which is very fast.
tf_to_count_map = dis_updated.set_index('TF')['Overlapping_Sites_Count']

print("\n--- Created lookup mapping (partial) ---")
print(tf_to_count_map.head())


# --- Step 2: Define a function to calculate the sum of interaction partners' counts ---
def get_interaction_partners_sum(current_tf, interaction_df, lookup_map):
    """
    Calculates the sum of Overlapping_Sites_Count for all interaction partners of a given TF.
    
    Parameters:
    - current_tf: The name of the current TF to calculate (str)
    - interaction_df: DataFrame containing interaction relationships (filtered_df)
    - lookup_map: Mapping of TF names to count values (pd.Series)
    """
    # Find partners where current_tf appears in the 'tf1' column (partners are in 'tf2')
    partners1 = interaction_df[interaction_df['tf1'] == current_tf]['tf2']
    
    # Find partners where current_tf appears in the 'tf2' column (partners are in 'tf1')
    partners2 = interaction_df[interaction_df['tf2'] == current_tf]['tf1']
    
    # Combine both sets of partners and remove duplicates
    all_partners = pd.concat([partners1, partners2]).unique()
    
    # If there are no partners, return 0
    if len(all_partners) == 0:
        return 0
        
    # Use .get(partner, 0) to safely retrieve the count from the map. 
    # If the partner is not in the map (e.g., it wasn't measured), it counts as 0.
    # Then sum the count values of all partners.
    total_sum = sum(lookup_map.get(partner, 0) for partner in all_partners)
    
    return total_sum

# --- Step 3: Apply the function to each row of dis_updated ---
# Call our function for every TF in the 'TF' column and store the result in a new column
dis_updated['Interaction_Partners_Count_Sum'] = dis_updated['TF'].apply(
    get_interaction_partners_sum,
    args=(filtered_df, tf_to_count_map) # Pass additional arguments
)


# --- Final Result ---
print("\n--- Final result after adding the new column ---")
print(dis_updated)

In [ ]:
dis_updated

In [ ]:
dis_updated[dis_updated['Overlapping_Sites_Count'] == 0]

In [ ]:
dis_updated[dis_updated['Interaction_Partners_Count_Sum'] == 0]

In [ ]:
condition1 = dis_updated['Overlapping_Sites_Count'] == 0

condition2 = dis_updated['Interaction_Partners_Count_Sum'] == 0

zero_rows = dis_updated[condition1 & condition2]

print("\n--- Rows where both selected columns are zero ---")
print(zero_rows)